In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
"""
Bước 1: Data Loader
Dự án: Phân tích và dự đoán tỉ lệ nghỉ việc (IBM HR Attrition Enhanced)
bằng mô hình cây quyết định.

Chức năng:
    - Nạp file CSV vào DataFrame
    - Kiểm tra kích thước, kiểu dữ liệu, giá trị thiếu, dòng trùng lặp
    - Kiểm tra biến mục tiêu Attrition (phân phối, mức độ mất cân bằng)

Lưu ý: bước này CHỈ đọc và kiểm tra, không sửa đổi dữ liệu.
Việc làm sạch thuộc về bước Data Cleaner.
"""

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# ====== CẤU HÌNH ======
DATA_PATH = "IBM_HR_Attrition_Enhanced.csv"  # đường dẫn local (dùng với load_data)
DRIVE_DATA_PATH = (
    "/content/drive/My Drive/Colab Notebooks/Project/IBM_HR_Attrition_Enhanced.csv"
)  # đường dẫn trên Google Drive (dùng với load_data_from_drive)
TARGET_COL = "Attrition"


# ====== 1. NẠP DỮ LIỆU ======
def load_data(path: str = DATA_PATH) -> pd.DataFrame:
    """Đọc file CSV vào DataFrame, thử nhiều encoding phổ biến."""
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {file_path.resolve()}")

    last_error = None
    for encoding in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"[OK] Đã nạp '{file_path.name}' (encoding={encoding})")
            return df
        except UnicodeDecodeError as e:
            last_error = e
    raise last_error


def load_data_from_drive(path: str = DRIVE_DATA_PATH) -> pd.DataFrame:
    """Mount Google Drive trong Colab rồi đọc CSV từ thư mục dự án.

    Đọc TOÀN BỘ các cột (không dùng usecols) để các bước sau có đủ dữ liệu.
    """
    try:
        from google.colab import drive
    except ImportError as e:
        raise RuntimeError(
            "Hàm này chỉ chạy được trong Google Colab. "
            "Nếu chạy trên máy local, hãy dùng load_data()."
        ) from e

    if not Path("/content/drive/My Drive").exists():
        drive.mount("/content/drive")
    else:
        print("[INFO] Google Drive đã được mount.")

    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file trên Drive: {file_path}\n"
            "Kiểm tra lại tên thư mục/tên file (phân biệt hoa thường)."
        )

    df = pd.read_csv(file_path, engine="python")
    print(f"[OK] Đã nạp '{file_path.name}' từ Google Drive")
    return df


# ====== 2. KIỂM TRA CẤU TRÚC ======
def check_shape(df: pd.DataFrame) -> None:
    rows, cols = df.shape
    print("\n" + "=" * 60)
    print("KÍCH THƯỚC DỮ LIỆU")
    print("=" * 60)
    print(f"Số dòng (nhân viên): {rows:,}")
    print(f"Số cột (thuộc tính): {cols}")
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Bộ nhớ sử dụng     : {mem_mb:.2f} MB")


def check_dtypes(df: pd.DataFrame) -> None:
    print("\n" + "=" * 60)
    print("KIỂU DỮ LIỆU")
    print("=" * 60)

    summary = pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "n_unique": df.nunique(),
            "n_missing": df.isna().sum(),
            "pct_missing": (df.isna().mean() * 100).round(2),
        }
    )
    print(summary.to_string())

    num_cols = df.select_dtypes(include="number").columns.tolist()
    cat_cols = df.select_dtypes(exclude="number").columns.tolist()
    print(f"\nCột số      ({len(num_cols)}): {num_cols}")
    print(f"Cột phân loại ({len(cat_cols)}): {cat_cols}")


def check_quality(df: pd.DataFrame) -> None:
    """Kiểm tra nhanh: dòng trùng, giá trị thiếu, cột hằng số."""
    print("\n" + "=" * 60)
    print("CHẤT LƯỢNG DỮ LIỆU (chỉ thống kê, chưa xử lý)")
    print("=" * 60)

    n_dup = df.duplicated().sum()
    n_missing = int(df.isna().sum().sum())
    print(f"Dòng trùng lặp hoàn toàn: {n_dup}")
    print(f"Tổng số ô thiếu giá trị : {n_missing}")

    constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print(f"Cột chỉ có 1 giá trị (không có tác dụng phân tách): {constant_cols or 'không có'}")

    # Cột gần như là mã định danh (mỗi dòng một giá trị khác nhau)
    id_like = [
        c for c in df.columns
        if df[c].nunique() == len(df) and c != TARGET_COL
    ]
    print(f"Cột giống mã định danh (nên loại ở bước cleaner): {id_like or 'không có'}")


# ====== 3. KIỂM TRA BIẾN MỤC TIÊU ======
def check_target(df: pd.DataFrame, target: str = TARGET_COL) -> None:
    print("\n" + "=" * 60)
    print(f"BIẾN MỤC TIÊU: {target}")
    print("=" * 60)

    if target not in df.columns:
        similar = [c for c in df.columns if target.lower() in c.lower()]
        raise KeyError(
            f"Không có cột '{target}'. Cột gần giống: {similar or 'không có'}. "
            f"Danh sách cột: {df.columns.tolist()}"
        )

    y = df[target]
    print(f"Kiểu dữ liệu     : {y.dtype}")
    print(f"Giá trị thiếu    : {y.isna().sum()}")
    print(f"Các giá trị duy nhất: {y.dropna().unique().tolist()}")

    counts = y.value_counts(dropna=False)
    pcts = y.value_counts(normalize=True, dropna=False) * 100
    dist = pd.DataFrame({"so_luong": counts, "ty_le_%": pcts.round(2)})
    print("\nPhân phối lớp:")
    print(dist.to_string())

    # Mức độ mất cân bằng
    vc = y.value_counts()
    if len(vc) >= 2:
        ratio = vc.max() / vc.min()
        print(f"\nTỉ lệ lớp đa số / lớp thiểu số = {ratio:.2f} : 1")
        if ratio >= 3:
            print("=> Dữ liệu MẤT CÂN BẰNG rõ rệt: cần bước Imbalance Handler, "
                  "và nên dùng F1/Recall/PR-AUC thay vì chỉ Accuracy.")
        elif ratio >= 1.5:
            print("=> Dữ liệu mất cân bằng nhẹ: nên cân nhắc class_weight hoặc stratify.")
        else:
            print("=> Dữ liệu tương đối cân bằng.")


# ====== 4. HÀM TỔNG HỢP ======
def run_data_loader(path: str = DRIVE_DATA_PATH, target: str = TARGET_COL) -> pd.DataFrame:
    """Chạy toàn bộ bước 1 và trả về DataFrame gốc (chưa chỉnh sửa)."""
    df = load_data_from_drive(path)

    print("\n5 dòng đầu:")
    print(df.head().to_string())

    check_shape(df)
    check_dtypes(df)
    check_quality(df)
    check_target(df, target)

    print("\n[HOÀN TẤT] Bước 1 - Data Loader.")
    return df


if __name__ == "__main__":
    df = run_data_loader()

[INFO] Google Drive đã được mount.
[OK] Đã nạp 'IBM_HR_Attrition_Enhanced.csv' từ Google Drive

5 dòng đầu:
   Age Attrition     BusinessTravel  DailyRate              Department  DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  EnvironmentSatisfaction  Gender  HourlyRate  JobInvolvement  JobLevel                JobRole  JobSatisfaction MaritalStatus  MonthlyIncome  MonthlyRate  NumCompaniesWorked Over18 OverTime  PercentSalaryHike  PerformanceRating  RelationshipSatisfaction  StandardHours  StockOptionLevel  TotalWorkingYears  TrainingTimesLastYear  WorkLifeBalance  YearsAtCompany  YearsInCurrentRole  YearsSinceLastPromotion  YearsWithCurrManager AgeGroup             TenureGroup MonthlyIncomeBracket  OverallSatisfactionIndex
0   41       Yes      Travel_Rarely       1102                   Sales                 1          2  Life Sciences              1               1                        2  Female          94               3         2        Sales Executiv